# 方法1、直接呼び出し

In [3]:
from langchain_core.messages.tool import tool_call
from langchain_core.tools import tool

 
@tool
def get_weather(city: str):
    '''
    指定した都市の天気情報を取得する
    引数：
        city:都市名、例："北京"、"上海"
        
    戻り値：
        天気情報の文字列
    '''
    return city + "、晴れ、気温15度"

In [4]:
get_weather.invoke("北京")

'北京晴天，温度15'

# モデルを使った呼び出し

In [5]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL")

model = init_chat_model(
    model=DEEPSEEK_MODEL,
    api_key=DEEPSEEK_API_KEY,
)


In [6]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """
    指定した都市の天気を取得する
    """
    # ここに実装
    return "晴れ、気温15度"


model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("東京の天気はどうですか？")
# AIがツールを呼び出そうとしているか確認
if response.tool_calls:
    print("AIがツールを呼び出そうとしています：", response.tool_calls)
else:
    print("AIが直接回答：", response.content)

AIがツールを呼び出そうとしています： [{'name': 'get_weather', 'args': {'city': '東京'}, 'id': 'call_00_dVaw7ItzSdacwGxp9h281855', 'type': 'tool_call'}]


# メッセージの流れからツール呼び出しを見る

In [7]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain_core.tools import tool

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL")

model = init_chat_model(
    model=DEEPSEEK_MODEL,
    api_key=DEEPSEEK_API_KEY,
)




In [8]:
from langchain_core.tools import tool
from rich import print
from langchain.messages import HumanMessage, ToolMessage


@tool
def get_weather(city: str):
    """天気を取得するツール"""
    return f"{city}は晴れです~"


# モデルとツールをバインド
model_with_tools = model.bind_tools([get_weather])

# メッセージリストを宣言
messages = [
    HumanMessage("今日の北京の天気はどうですか")
]

# モデルがツール呼び出しリクエストを生成
response = model_with_tools.invoke(messages)

# メッセージリストに AIMessage を追加
messages.append(response)

# rprint(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 大規模言語モデルとAgentの主な違いは：大規模言語モデルは自らツールを呼び出さないため、ここでは明示的にツールを呼び出す必要がある。
        # 戻り値は ToolMessage 型のメッセージで、メッセージリストに追加する
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)

print("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>

=====================> messages <=====================

================================ Human Message =================================

今日の北京の天気はどうですか
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_zzvQ9Bkgmv9XMtDLKu1z9713)
 Call ID: call_00_zzvQ9Bkgmv9XMtDLKu1z9713
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京は晴れです~


=====================> messages <=====================

final_response: 
content='今日の北京の天気は**晴れ**です！☀️\n\n良い一日をお過ごしください。何か他にお手伝いできることはありますか？
' additional_kwargs={'refusal': None, 'reasoning_content': 
'ツールから「北京は晴れです」という結果が返ってきました。ユーザーに今日の北京の天気を伝えます。'} 
response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 375, 'total_tokens': 444, 
'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 31, 
'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 
'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 119}, 'model_provider': 'deepseek', 'model_name': 
'deepseek-v4-pro', 'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402', 'id': 
'407ec842-0915-43f1-8c5a-96ab70962104', 'finish_reason': 'stop', 'logprobs': None} 
id='lc_run--019fc16e-c1a6-7563-982b-8b822b39cbdb-0' tool_calls=[] invalid_tool_calls=[] 
usage_metadata={'input_tokens': 375, 'output_tokens': 69, 'total_tokens': 444, 'input_token_details': 
{'cache_read': 256}, 'output_token_details': {'reasoning': 31}}